In [48]:
import hashlib
import random
import time
import urllib
from json import loads

from trafilatura import fetch_url, extract
from w3lib.url import canonicalize_url

# Collection of specific blog post/article URLs across different themes
urls = {
    "lifestyle": [
        "https://www.vogue.com/article/spring-2025-fashion-trends",
    ]
}


def scrape_url(url):
    """Scrape a single blog post/article URL"""
    try:
        downloaded = fetch_url(url=url)
        if not downloaded:
            return None

        result = extract(downloaded, output_format="json", include_comments=False, include_images=False,
                         include_links=False, with_metadata=True)

        if not result:
            return None

        result = loads(result)

        payload = {
            "image": result.get("image", ""),
            "url": result.get("source", url),
            "tags": result.get("tags", "").split(", ") if result.get("tags") else [],
            "title": result.get("title", ""),
            "description": result.get("description", ""),
            "content": result.get("raw_text", ""),
            "date": result.get("date", ""),
            "author": result.get("author", "")
        }
        return payload
    except Exception as e:
        print(f"Error scraping {url}: {str(e)}")
        return None


# Scrape all articles
scraped_data = []
for theme, theme_urls in urls.items():
    print(f"\nScraping {theme.upper()} articles...")
    for url in theme_urls:
        print(f"  Processing: {url}")
        data = scrape_url(url)
        if data:
            data["theme"] = theme
            scraped_data.append(data)

        time.sleep(random.uniform(1, 3))

print(f"\n✓ Successfully scraped {len(scraped_data)} articles")



Scraping LIFESTYLE articles...
  Processing: https://www.vogue.com/article/spring-2025-fashion-trends
  Processing: https://variety.com/2025/tv/news/jimmy-kimmel-late-night-return-review-suspension-1236527655/

✓ Successfully scraped 2 articles


In [92]:
url = "https://www.vogue.com/article/9-outdated-fashion-pieces-vogue-editors-still-love"
downloaded = fetch_url(url=url)
result = extract(downloaded, output_format="json", include_comments=False, include_images=False, include_links=False, with_metadata=True)

In [93]:
result = loads(result)

In [94]:
result

{'title': '9 Outdated Fashion Trends That Vogue Editors Still Love in 2025',
 'author': 'Christian Allaire',
 'hostname': 'vogue.com',
 'date': '2025-03-06',
 'fingerprint': '3e709b75d39f323d',
 'id': None,
 'license': None,
 'comments': '',
 'raw_text': 'All products featured on Vogue are independently selected by our editors. However, we may receive compensation from retailers and/or from purchases of products through these links. Back in January, Timothée Chalamet stepped out in New York City wearing a serious throwback piece: An Alexander McQueen skull scarf. Having reached its peak in the late 2000s during the indie sleaze movement, that edgy skull-print scarf was worn by all the It girls at the time—from Lindsay Lohan to Mary-Kate Olsen—and is now considered a designer relic of the past. Outdated, even. But as Chalamet’s revival of it proves, who says something has to be out-of-style? When styled right, even the most forgotten fashion pieces can still feel fresh. The fashion mome

In [49]:
scraped_data[0]

{'image': 'https://assets.vogue.com/photos/67c9df7ca1f72dd92fd6aee3/16:9/w_1280,c_limit/holding-rtw.png',
 'url': 'https://www.vogue.com/article/spring-2025-fashion-trends',
 'tags': [],
 'title': 'The Top Spring/Summer 2025 Fashion Trends You Can Shop and Wear Now',
 'description': '',
 'content': 'All products featured on Vogue are independently selected by our editors. However, we may earn affiliate revenue on this article and commission when you buy something. While strong notions of femininity prevailed in the spring/summer collections, the spring/summer 2025 fashion trends we’re predicting to take over the fashion landscape are less ethereal and dreamy as they are practical and empowering. Rooted in a sense of “soft power,” which Vogue’s Laird Borelli-Persson connected to designers encouraging “an openness to a sense of wonder” in her analysis of the shows, these trends can be embraced in more ways than one. And key themes can already be incorporated into your wardrobe. Vogue’s T

KEYWORD AND NAMED ENTITY EXTRACTION

In [102]:
from keybert import KeyBERT

# KeyBERT will use "all-MiniLM-L6-v2" by default
kw_model = KeyBERT()

In [109]:


doc = scraped_data[0]["content"]

from itertools import chain

# Extract both unigram and bigram keywords
keywords = kw_model.extract_keywords(doc, keyphrase_ngram_range=(1, 1), top_n=20, stop_words="english")
keywords1 = kw_model.extract_keywords(doc, keyphrase_ngram_range=(1, 2), top_n=20, stop_words="english")

# Combine and keep only the highest scoring keyword occurrences
all_keywords = {}
for keyword, score in chain(keywords, keywords1):
    all_keywords[keyword] = max(score, all_keywords.get(keyword, 0))

print(all_keywords)

{'vogue': 0.608, 'fashion': 0.571, 'dresses': 0.4789, 'wardrobe': 0.4753, 'outerwear': 0.4378, 'dress': 0.4342, 'skirt': 0.4161, 'styles': 0.4123, 'skirts': 0.4115, 'dressing': 0.4005, 'tailored': 0.3575, 'trends': 0.3518, 'tailoring': 0.3516, 'shirts': 0.3478, 'wearing': 0.3417, 'designers': 0.3416, 'styled': 0.3396, 'wear': 0.3388, 'trend': 0.3333, 'denim': 0.33, 'vogue spring': 0.6652, 'wardrobe vogue': 0.66, 'fashion trends': 0.6556, 'fashion ongoing': 0.6273, 'featured vogue': 0.5965, 'vogue independently': 0.5949, 'predicting fashion': 0.5821, 'summertime dress': 0.5778, 'fashion mood': 0.5772, 'exemplifies fashion': 0.5712, '2025 fashion': 0.5522, 'fashion landscape': 0.5482, 'spring outerwear': 0.5328, 'vogue mark': 0.5192, 'fashion pov': 0.5129, 'skirts dresses': 0.5085, 'season outerwear': 0.5061, 'minimalist wardrobe': 0.505}


In [96]:
 !python -m spacy download en_core_web_sm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [51]:
import spacy

nlp = spacy.load("en_core_web_sm")

text = doc

doc_trf = nlp(text)

entities = dict()
for ent in doc_trf.ents:
    key = ent.text.lower()
    if key not in entities:
        entities[key] = (ent.text, ent.label_)
entities = list(entities.values())


In [52]:
entities

[('Vogue', 'PERSON'),
 ('the spring/summer', 'DATE'),
 ('the spring/summer 2025', 'DATE'),
 ('Laird Borelli-Persson', 'PERSON'),
 ('Spring 2025', 'DATE'),
 ('2,490', 'MONEY'),
 ('The Bohemian Blouse', 'ORG'),
 ('Banana Republic', 'GPE'),
 ('100', 'MONEY'),
 ('Prada', 'ORG'),
 ('2,950', 'MONEY'),
 ('The Vintage-Inspired Print: Dries Van Noten', 'ORG'),
 ('620', 'MONEY'),
 ('The Checked Top: Simkhai Calliope', 'WORK_OF_ART'),
 ('495', 'MONEY'),
 ('The Frankie Shop Bea', 'ORG'),
 ('345', 'MONEY'),
 ('Diotima Darliston', 'PERSON'),
 ('1,095', 'MONEY'),
 ('The New Proportions Mini', 'ORG'),
 ('Alaïa', 'PERSON'),
 ('3,900', 'MONEY'),
 ('148', 'CARDINAL'),
 ('1,798', 'MONEY'),
 ('Aligne Omari', 'PERSON'),
 ('90', 'MONEY'),
 ('Tory Burch', 'PERSON'),
 ('2,898', 'MONEY'),
 ('the season ahead', 'DATE'),
 ('Bottega', 'PERSON'),
 ('Suede', 'ORG'),
 ('last fall', 'DATE'),
 ('first', 'ORDINAL'),
 ('spring', 'DATE'),
 ('this spring', 'DATE'),
 ('Van Noten', 'PERSON'),
 ('Gucci', 'PERSON'),
 ('Miu Miu

TOPIC CLASSIFICATION WITH ZERO-SHOT LEARNING

In [53]:

import json

with open("../data/iab_taxonomy.json", "r") as f:
    iab_taxonomy = json.load(f)

In [54]:
import os
from typing import List, Dict, Any, Tuple
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sentence_transformers import SentenceTransformer

# 0) Runtime setup
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")


def _device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


DEVICE = _device()
DTYPE = torch.float16 if DEVICE.type in {"cuda", "mps"} else torch.float32

# 1) Choose model (DeBERTa: better accuracy/512t; BART: 1024t -> fewer chunks)
MODEL_ID = "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"  # or "facebook/bart-large-mnli"

# Build once
_zs_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
_zs_tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
_zs_model.to(DEVICE).eval()
classifier = pipeline(
    "zero-shot-classification",
    model=_zs_model,
    tokenizer=_zs_tok,
    device=0 if DEVICE.type == "cuda" else (-1 if DEVICE.type == "cpu" else DEVICE),
    # pipeline accepts torch.device in recent HF
    batch_size=8
)
tokenizer = classifier.tokenizer

# Fast embedder to shortlist labels per tier
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

Device set to use mps


In [55]:


# 2) Chunking with low overlap (fewer chunks)
def chunk_text_by_tokens(text: str, max_tokens: int | None = None, overlap: int = 32) -> List[str]:
    limit = tokenizer.model_max_length if max_tokens is None else min(max_tokens, tokenizer.model_max_length)
    usable = max(64, limit - 16)  # leave space for specials
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= usable:
        return [text]
    stride = max(1, usable - overlap)
    chunks = []
    for start in range(0, len(ids), stride):
        chunk_ids = ids[start:start + usable]
        if not chunk_ids:
            break
        chunks.append(tokenizer.decode(chunk_ids, skip_special_tokens=True))
        if start + usable >= len(ids):
            break
    return chunks


# 3) Shortlist labels with embeddings to cut NLI work by ~10x
def shortlist_labels(text_chunks: List[str], labels: List[str], top_k: int = 8) -> List[str]:
    if len(labels) <= top_k:
        return labels
    # Encode once per tier
    chunk_vecs = embedder.encode(text_chunks, batch_size=32, normalize_embeddings=True, convert_to_numpy=True)
    label_vecs = embedder.encode(labels, batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    sims = label_vecs @ chunk_vecs.T  # [L, C]
    per_label = sims.max(axis=1)  # best-chunk similarity for each label
    idx = np.argsort(-per_label)[:top_k]
    return [labels[i] for i in idx]


# 4) Score a single tier with chunking + label shortlisting + aggregation
def score_tier(
        text: str,
        candidate_labels: List[str],
        multi_label: bool,
        hypothesis_template: str,
        aggregate: str = "mean",
        batch_size: int = 8,
        max_tokens: int | None = None,
        overlap: int = 32,
        shortlist_top_k: int = 8
) -> Dict[str, float]:
    chunks = chunk_text_by_tokens(text, max_tokens=max_tokens, overlap=overlap)
    labels = shortlist_labels(chunks, candidate_labels, top_k=min(shortlist_top_k, len(candidate_labels)))
    # Batch over all chunks in one call
    results = classifier(
        sequences=chunks if len(chunks) > 1 else chunks[0],
        candidate_labels=labels,
        multi_label=multi_label,
        hypothesis_template=hypothesis_template,
        batch_size=batch_size
    )
    if isinstance(results, dict):
        results = [results]

    per_label_scores: Dict[str, List[float]] = {lab: [] for lab in labels}
    for out in results:
        for lab, s in zip(out["labels"], out["scores"]):
            per_label_scores[lab].append(float(s))

    def reduce(vals: List[float]) -> float:
        if not vals:
            return 0.0
        return max(vals) if aggregate == "max" else (sum(vals) / len(vals))

    # Produce scores only for shortlisted labels; others are zero
    scores = {lab: reduce(v) for lab, v in per_label_scores.items()}
    for lab in candidate_labels:
        scores.setdefault(lab, 0.0)
    return scores


def hierarchical_zero_shot(text: str,
                           hierarchy: List[Dict[str, Any]],
                           multi_label_per_tier: bool = True,
                           tier_threshold: float = 0.5,
                           tier_top_k: int = 2,
                           hypothesis_template: str = "This text is about {}.",
                           score_aggregate: str = "mean",
                           max_tokens: int | None = None,
                           overlap: int = 32,
                           batch_size: int = 8,
                           shortlist_top_k: int = 8,
                           return_top_paths: int = 5
                           ) -> List[Dict[str, Any]]:
    branches: List[Tuple[List[Dict[str, Any]], List[str], List[str], float]] = [(hierarchy, [], [], 1.0)]
    completed: List[Tuple[List[str], str, float]] = []

    while branches:
        next_branches: List[Tuple[List[Dict[str, Any]], List[str], List[str], float]] = []
        for nodes, path, ids, cum_score in branches:
            if not nodes:
                last_id = ids[-1] if ids else ""
                completed.append((path, last_id, cum_score))
                continue

            labels = [n.get("name", "") for n in nodes if n.get("name")]
            if not labels:
                last_id = ids[-1] if ids else ""
                completed.append((path, last_id, cum_score))
                continue

            scores = score_tier(
                text=text,
                candidate_labels=labels,
                multi_label=multi_label_per_tier,
                hypothesis_template=hypothesis_template,
                aggregate=score_aggregate,
                batch_size=batch_size,
                max_tokens=max_tokens,
                overlap=overlap,
                shortlist_top_k=shortlist_top_k
            )

            ranked = sorted(((lab, scores.get(lab, 0.0)) for lab in labels), key=lambda x: x[1], reverse=True)
            if multi_label_per_tier:
                chosen = [(lab, s) for lab, s in ranked if s >= tier_threshold][:tier_top_k]
            else:
                lab, s = ranked[0]
                chosen = [(lab, s)] if s >= tier_threshold else []

            if not chosen:
                last_id = ids[-1] if ids else ""
                completed.append((path, last_id, cum_score))
                continue

            for lab, s in chosen:
                node = next((n for n in nodes if n.get("name") == lab), None)
                if node:
                    node_id = node.get("id", "")
                    child_nodes = node.get("children", [])
                    next_branches.append((child_nodes, path + [lab], ids + [node_id], cum_score * s))

        if not next_branches:
            break
        branches = next_branches

    for _, path, ids, cum_score in branches:
        last_id = ids[-1] if ids else ""
        completed.append((path, last_id, cum_score))

    completed = sorted(completed, key=lambda x: x[2], reverse=True)
    ans = []
    seen_iab_ids = set()
    for p, iab_id, s in completed:
        if iab_id in seen_iab_ids:
            continue
        ans.append({"category": " > ".join(p), "iab_id": iab_id, "score": s})
        seen_iab_ids.add(iab_id)
    return ans[:return_top_paths]



In [57]:
paths = hierarchical_zero_shot(
    text=scraped_data[0]["content"],
    hierarchy=iab_taxonomy,
    multi_label_per_tier=True,
    tier_threshold=0.5,
    tier_top_k=2,
    hypothesis_template="The topic of this text is {}.",
    score_aggregate="mean",  # or "max"
    max_tokens=300,  # use model limit
    overlap=96,
    batch_size=8,
    return_top_paths=5
)
print(paths)

[{'category': 'Style & Fashion > Fashion Trends', 'iab_id': '577', 'score': 0.99353398762584}, {'category': "Style & Fashion > Women's Fashion > Women's Clothing", 'iab_id': '566', 'score': 0.776141775275761}, {'category': 'Shopping', 'iab_id': '473', 'score': 0.604876471683383}]


EMBEDDING AND CHUNKING FOR SEMANTIC

In [34]:
import numpy as np
from sentence_transformers import SentenceTransformer, util

# Load model once (not every call)
model = SentenceTransformer('all-MiniLM-L6-v2')


def sent_tokenize(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]


def semantic_chunker(content: str, breakpoint_percentile_threshold: float = 90.0) -> list[str]:
    """
    Splits text segments into semantically cohesive chunks.
    Breaks where similarity between adjacent segments falls below the given percentile.

    Args:
        content: Text to be split.
        breakpoint_percentile_threshold: Percentile (0-100) of similarity scores.
                                         Higher values = fewer splits (stricter).

    Returns:
        List of text chunks.
    """
    texts: list[str] = sent_tokenize(content)
    if not texts:
        return []
    if len(texts) == 1:
        return texts

    # Compute embeddings
    embeddings = model.encode(texts, convert_to_tensor=True)

    # Compute similarity between consecutive sentences
    similarities = util.cos_sim(embeddings[:-1], embeddings[1:]).diagonal().cpu().numpy()

    # Compute similarity threshold
    threshold = np.percentile(similarities, breakpoint_percentile_threshold)

    # Determine where to split
    split_indices = [i for i, sim in enumerate(similarities) if sim < threshold]

    # Build chunks
    chunks = []
    start = 0
    for idx in split_indices:
        chunks.append(" ".join(texts[start:idx + 1]))
        start = idx + 1

    if start < len(texts):
        chunks.append(" ".join(texts[start:]))

    return chunks

In [87]:
chunks = semantic_chunker(content=result["title"] + " . " + result["raw_text"], breakpoint_percentile_threshold=50.0)
print(len(chunks))

40


In [88]:
chunks

['9 Outdated Fashion Trends That Vogue Editors Still Love in 2025 . All products featured on Vogue are independently selected by our editors.',
 'However, we may receive compensation from retailers and/or from purchases of products through these links.',
 'Back in January, Timothée Chalamet stepped out in New York City wearing a serious throwback piece: An Alexander McQueen skull scarf. Having reached its peak in the late 2000s during the indie sleaze movement, that edgy skull-print scarf was worn by all the It girls at the time—from Lindsay Lohan to Mary-Kate Olsen—and is now considered a designer relic of the past.',
 'Outdated, even. But as Chalamet’s revival of it proves, who says something has to be out-of-style? When styled right, even the most forgotten fashion pieces can still feel fresh. The fashion moment got us thinking: Being trendy is seriously overrated, anyways. Sure, updated silhouettes and brand-new pieces can inject your wardrobe with a sense of newness—but shouldn’t 

In [91]:
from sentence_transformers import SentenceTransformer, util

def sent_tokenize(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]

def semantic_chunk(text, similarity_threshold=0.4, max_chunk_sentences=5):
    # Load a small, efficient embedding model
    model = SentenceTransformer('all-MiniLM-L6-v2')

    sentences = sent_tokenize(text)

    # Step 2: Compute embeddings
    embeddings = model.encode(sentences, convert_to_tensor=True)

    chunks = []
    current_chunk = [sentences[0]]
    current_sims = []

    for i in range(1, len(sentences)):
        sim = util.cos_sim(embeddings[i-1], embeddings[i]).item()
        current_sims.append(sim)

        # Compute running average similarity in current chunk
        avg_sim = sum(current_sims) / len(current_sims)

        if avg_sim < similarity_threshold or len(current_chunk) >= max_chunk_sentences:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
            current_sims = []
        else:
            current_chunk.append(sentences[i])

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# Example Usage
text = """
Artificial intelligence is transforming industries around the world.
Machine learning and deep learning are key components of AI.
They are used in healthcare, finance, and transportation.
On the other hand, there are concerns about bias and fairness.
Regulators are trying to ensure responsible AI development.
"""

chunks = semantic_chunk(text)
for i, c in enumerate(chunks, 1):
    print(f"Chunk {i}:\n{c}\n")

Chunk 1:
Artificial intelligence is transforming industries around the world. Machine learning and deep learning are key components of AI.

Chunk 2:
They are used in healthcare, finance, and transportation.

Chunk 3:
On the other hand, there are concerns about bias and fairness.

Chunk 4:
Regulators are trying to ensure responsible AI development.



In [40]:
ans = embedder.encode(chunks, convert_to_numpy=True)

In [42]:
len(ans[30])

384

In [63]:
def generate_hash_and_url(url: str) -> tuple[str, str]:
    canonical_url = canonicalize_url(url)
    print("Canonical URL: ", canonical_url)

    url_hash = hashlib.sha256(canonical_url.encode('utf-8')).hexdigest()
    return url_hash, canonical_url

In [67]:
generate_hash_and_url('https://www.vogue.com/article/9-outdated-fashion-pieces-vogue-editors-still-love')


Canonical URL:  https://www.vogue.com/article/9-outdated-fashion-pieces-vogue-editors-still-love


('8e052c7004d0f4683be026d7f1f5ea799e00437f455765aac68f7b3eb01e126e',
 'https://www.vogue.com/article/9-outdated-fashion-pieces-vogue-editors-still-love')

In [72]:
from typing import Literal
import hashlib
import urllib.parse
import urllib

def canonical_url(url: str) -> Literal[b""]:
    parsed = urllib.parse.urlparse(url)
    normalized = parsed._replace(
        scheme=parsed.scheme.lower(),
        netloc=parsed.netloc.lower(),
        path=parsed.path.rstrip('/')
    )
    return urllib.parse.urlunparse(normalized)

def url_id(url: str) -> str:
    normalized = canonical_url(url)
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

In [73]:
url_id('https://www.vogue.com/article/9-outdated-fashion-pieces-vogue-editors-still-love')


'8e052c7004d0f4683be026d7f1f5ea799e00437f455765aac68f7b3eb01e126e'

get the complete path and have a the probability of that happpeing, so multiply the P(T1)X P(T2).. P(TN)